# Phase 3 & 4: Deep Learning Modeling
This notebook trains both the LSTM baseline and the Transformer Temporal Encoder.

In [ ]:
import os
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('GPNCM'):
        !git clone https://github.com/TinevimboMusingadi/GPNCM.git
    %cd GPNCM
    PROJECT_DIR = '/content/GPNCM'
    print("Running in Colab, working directory is now GPNCM")
else:
    PROJECT_DIR = '..'
    print("Running locally")

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


In [ ]:
import sys
sys.path.append(PROJECT_DIR)

import pandas as pd
import yaml
import os
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from src.data_loader import make_sequences
from src.preprocessing import split_and_scale_data
from src.lstm_model import build_lstm
from src.transformer_model import build_transformer
from src.evaluate import evaluate_model
from utils.plot_utils import plot_training_curves, plot_predictions, plot_mape_comparison

# Load Config
with open(f'{PROJECT_DIR}/config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)

In [ ]:
# 1. Load and prepare Data
df = pd.read_parquet(f'{PROJECT_DIR}/data/ireland_clean.parquet')
feature_cols = config['data']['feature_cols']
target_idx = feature_cols.index(config['data']['target_col'])

train_scaled, val_scaled, test_scaled, scaler = split_and_scale_data(
    df, 
    feature_cols, 
    config['data']['target_col'],
    train_frac=config['data']['train_frac'],
    val_frac=config['data']['val_frac'],
    scaler_save_path=f'{PROJECT_DIR}/models/scaler.pkl'
)

n_past = config['data']['n_past']
n_future = config['data']['n_future']

X_train, y_train = make_sequences(train_scaled, n_past, n_future, target_idx)
X_val, y_val = make_sequences(val_scaled, n_past, n_future, target_idx)
X_test, y_test = make_sequences(test_scaled, n_past, n_future, target_idx)

print(f"Train pairs: {X_train.shape[0]}, Val pairs: {X_val.shape[0]}, Test pairs: {X_test.shape[0]}")

In [ ]:
# 2. Train Improved LSTM Baseline
model_lstm = build_lstm(n_past, len(feature_cols), units=config['lstm']['units'], 
                        dropout_rate=config['lstm']['dropout'], l2_lambda=float(config['lstm']['l2']))

callbacks_lstm = [
    EarlyStopping(monitor='val_loss', patience=config['lstm']['patience'], restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6),
    ModelCheckpoint(f'{PROJECT_DIR}/models/lstm_best.keras', monitor='val_loss', save_best_only=True)
]

history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config['lstm']['epochs'],
    batch_size=config['lstm']['batch_size'],
    callbacks=callbacks_lstm,
    verbose=1
)

plot_training_curves(history_lstm, f'{PROJECT_DIR}/outputs/lstm_training_curves.png', title='LSTM')

In [ ]:
# 3. Train Transformer Temporal Encoder
model_tf = build_transformer(n_past, len(feature_cols), 
                             d_model=config['transformer']['d_model'], 
                             num_heads=config['transformer']['num_heads'],
                             ff_dim=config['transformer']['ff_dim'],
                             num_blocks=config['transformer']['num_blocks'],
                             dropout_rate=config['transformer']['dropout'])

callbacks_tf = [
    EarlyStopping(monitor='val_loss', patience=config['transformer']['patience'], restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6),
    ModelCheckpoint(f'{PROJECT_DIR}/models/transformer_best.keras', monitor='val_loss', save_best_only=True)
]

history_tf = model_tf.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=config['transformer']['epochs'],
    batch_size=config['transformer']['batch_size'],
    callbacks=callbacks_tf,
    verbose=1
)

plot_training_curves(history_tf, f'{PROJECT_DIR}/outputs/transformer_training_curves.png', title='Transformer')

# Phase 5: Evaluation & Results Comparison

In [ ]:
y_true, y_pred_lstm, lstm_metrics = evaluate_model(model_lstm, X_test, y_test, scaler, target_idx, len(feature_cols), name="LSTM v2")
_, y_pred_tf, tf_metrics = evaluate_model(model_tf, X_test, y_test, scaler, target_idx, len(feature_cols), name="Transformer")

plot_predictions(y_true, y_pred_lstm, y_pred_tf, f'{PROJECT_DIR}/outputs/prediction_comparison.png', n_plot=500)

# Compare models
results_df = pd.DataFrame({
    'Model': ['LSTM v1 (2022)', 'LSTM v2 (improved)', 'Transformer (temporal)'],
    'MAPE (%)': [8.33, round(lstm_metrics['mape'], 2), round(tf_metrics['mape'], 2)],
})
plot_mape_comparison(results_df, f'{PROJECT_DIR}/outputs/mape_comparison.png')